# 02 · 构造、检查与重新运行 top-PC rotation

默认 p=500。先检查矩阵构造，再运行 DE 与小规模 MC；修改参数后重新 Run All。新实验的摘要、原始 fits、参数和日志缓存在 notebooks/outputs/top_pc_rotation/，配置不同使用不同目录。

A=FΣ½，b=Σ½β*，c=Σ⁻½β*。取 Q 为 span{b,c} 的正交基：
P=A₀QQᵀ，R=A₀−P，K=RRᵀ，V=K½W。
W 的行位于 tail PC band，正交于 b,c 且 WWᵀ=I。
A(t)=P+√(1−t)R+√tV，t=sin²θ。

左端 F₀ 是 unwhitened top-p projection；C=AAᵀ、h=Ab、q=Ac 保持不变，G=F Fᵀ 改变。这里只用 b 已足以约束预测；额外固定 c 是为隔离分母增长机制。

这些有限样本 MC 使用 Gaussian 特征和 DE 选择的固定 alpha，不是逐 trial RidgeCV；对真实非 Gaussian 图像，C,h 相同本身不保证训练分布相同。


In [ ]:
from pathlib import Path
import sys, time, json, hashlib, os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/accentuationpredrmt-matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/accentuationpredrmt-xdg-cache')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from threadpoolctl import threadpool_limits

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rmt_core").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or its notebooks directory.")
sys.path.insert(0, str(ROOT))
# Avoid excessive BLAS threads on shared machines.
blas_limit = threadpool_limits(limits=2)
from scripts import validate_vanhateren_top_pc_tail_rotations as experiment
OUT = ROOT / "notebooks" / "outputs" / "top_pc_rotation"
OUT.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)


In [ ]:
# Original run: P=500, N_TRIALS=300; or P=100, N_TRIALS=1000.
P = 500
N = 1000
RATIOS = np.array([0.01, 0.1, 1., 10.])
TAIL_START = 0.60
T = np.r_[0., np.geomspace(1e-6, 1., 61)]
ALPHAS = np.logspace(-4, 5, 281)
SEED = 20260901
N_TRIALS = 30                 # exploratory default
RUN_MC = True
MAX_PROJECTED_SECONDS = 120   # raise deliberately for larger experiments
FORCE = False
assert N_TRIALS >= 2
with np.load(experiment.SPECTRUM_PATH) as data:
    s = data["eigenvalues"].astype(float)
    beta = data["beta_proj"].astype(float)
family = experiment.build_top_pc_tail_family(
    s, beta, P, T, TAIL_START, SEED + P)
print("Signal captured:", family["represented_signal"] / family["signal"])
print("trace G endpoints:", family["trace_g"][[0, -1]])


## 显式查看 P、R、V（避免构造 d×d 投影矩阵）

这里的 P_map 是矩阵 P，P 是特征数。每个矩阵形状为 p×d；p=500 时每个 float64 矩阵约 40 MB。验证单个 t 足够随意检查；可改变 CHECK_T 重跑本格。


In [ ]:
Q = family["constraint_basis"]
A0 = np.zeros((P, len(s)))
A0[np.arange(P), np.arange(P)] = np.sqrt(s[:P])
P_map = family["left_factor"] @ Q.T
R = A0 - P_map
V = np.zeros_like(A0)
V[:, int(family["low_start"]):] = family["root_r"] @ family["w_low"]
CHECK_T = 1e-3
theta = np.arcsin(np.sqrt(CHECK_T))
A = P_map + np.cos(theta)*R + np.sin(theta)*V
F = A / np.sqrt(s)[None, :]
b, c = np.sqrt(s)*beta, beta/np.sqrt(s)
checks = {
    "AAᵀ − C": np.max(np.abs(A @ A.T - np.diag(s[:P]))),
    "Ab − h": np.max(np.abs(A @ b - s[:P]*beta[:P])),
    "Ac − q": np.max(np.abs(A @ c - beta[:P])),
    "VVᵀ − RRᵀ": np.max(np.abs(V @ V.T - R @ R.T)),
    "RVᵀ": np.max(np.abs(R @ V.T)),
}
display(pd.Series(checks, name="max absolute residual"))
assert max(checks.values()) < 1e-7
print("angle (degrees):", np.degrees(theta), "trace G:", np.sum(F**2))


## DE 选择 alpha，并检查均值、方差与 N/D

select_settings 用 n−1 样本的预测风险近似 LOOCV 来选择 alpha，再用 n 样本换算 κ。
DE 的二阶矩形式为 E[ŵŵᵀ] ≈ m mᵀ + diag(v)；
μN=qᵀm，μD=mᵀGm+Tr(G diag(v))。
leading DE 对非线性 ratio 使用矩之比；它不等于有限样本 ratio 的期望，近乎完美的端点 E_acc 尤其可能有相对误差。


In [ ]:
settings = experiment.select_settings(family, RATIOS, N, ALPHAS)
de = experiment.theory_metrics(family, settings, N)
display(pd.DataFrame(settings))
noise_index, angle_index = 0, -1
components = experiment.prediction_de_components(family, settings[noise_index], N)
m, v = components["mean"], components["variance"]
G = family["g_matrices"][angle_index]
mu_N = family["teacher_overlap"] @ m
mu_D = m @ G @ m + np.diag(G) @ v
display(pd.Series({"mu_N": mu_N, "mu_D": mu_D,
                   "slope_acc": mu_N/mu_D,
                   "R2_acc": 1-(mu_D/mu_N-1)**2,
                   "E_acc/S": (1-mu_N/mu_D)**2}))
assert np.allclose(mu_D, de["acc_denominator"][angle_index, noise_index])


## MC：先计时、报告 ETA，然后缓存

同一组特征空间 fits 作用于全部旋转位置。日志路径会打印；超出设置的时间预算会停止计算，可降低 trials 或调高 MAX_PROJECTED_SECONDS。
配置 hash 包括谱、teacher、代码和全部参数，避免误用旧缓存。

In [ ]:
config = dict(p=P, n=N, ratios=RATIOS.tolist(), tail_start=TAIL_START,
              t=T.tolist(), alphas=ALPHAS.tolist(), seed=SEED, trials=N_TRIALS,
              policy="fixed DE-LOOCV alpha; Gaussian design")
digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())
digest.update(s.tobytes()); digest.update(beta.tobytes())
for source in [Path(experiment.__file__), *sorted((ROOT/"rmt_core").glob("*.py"))]:
    digest.update(source.read_bytes())
run_dir = OUT / digest.hexdigest()[:16]
run_dir.mkdir(parents=True, exist_ok=True)
cache = run_dir / "mc.npz"
logger = experiment.configure_logger(run_dir / "progress.log")
print("Progress log:", run_dir / "progress.log")
mc = None
if RUN_MC:
    if cache.exists() and not FORCE:
        with np.load(cache) as data:
            mc = {key: data[key] for key in data.files}
        logger.info("Loaded existing MC cache %s", cache)
    else:
        start = time.perf_counter()
        pilot = experiment.simulate_shared_ridge(
            family, settings, N, 3, SEED + 10000 + P, progress=False)
        eta = (time.perf_counter()-start) * N_TRIALS / 3
        logger.info("Pilot projected runtime %.1f seconds for %d trials", eta, N_TRIALS)
        if eta > MAX_PROJECTED_SECONDS:
            raise RuntimeError(f"ETA {eta:.1f}s exceeds budget; adjust trials/budget.")
        mc = experiment.simulate_shared_ridge(
            family, settings, N, N_TRIALS, SEED + 20000 + P, progress=True)
        np.savez_compressed(cache, **mc)
        logger.info("MC finished; cached %s", cache)
(run_dir / "config.json").write_text(json.dumps(config, indent=2))
np.savez_compressed(run_dir / "de.npz", **de)


In [ ]:
if mc is not None:
    rows = experiment.build_summary_rows(
        {P: family}, {P: settings}, {P: de}, {P: mc}, N, {P: N_TRIALS})
else:
    rows = []
    for i, t in enumerate(T):
        for j, setting in enumerate(settings):
            rows.append(dict(p=P, tail_fraction=t,
                noise_signal_ratio=setting["noise_signal_ratio"], sigma=setting["sigma"],
                trace_g=family["trace_g"][i],
                effective_rank_g=family["effective_rank_g"][i],
                **{f"de_{key}": value[i,j] for key,value in de.items()}))
frame = pd.DataFrame(rows)
frame.to_csv(run_dir / "summary.csv", index=False)
display(frame.head())


In [ ]:
def plot_experiment(frame, p=500, cmap="viridis", marker_every=8,
                    r2_stat="median", figsize=(15, 8)):
    """Editable plot: lines = leading DE; circles = MC summaries."""
    data = frame[frame.p == p].copy()
    ratios = sorted(data.noise_signal_ratio.unique())
    colors = plt.get_cmap(cmap)(np.linspace(.06, .92, len(ratios)))
    fig, axes = plt.subplots(2, 3, figsize=figsize, sharex=True)
    base = data[np.isclose(data.noise_signal_ratio, ratios[0])].sort_values("tail_fraction")
    axes[0, 0].plot(base.tail_fraction, base.trace_g, color=".2")
    axes[0, 0].set(yscale="log", ylabel=r"$\mathrm{Tr}(FF^\top)$",
                   title=f"Unwhitened top-{p} PCs → tail")
    rank_ax = axes[0, 0].twinx()
    rank_ax.plot(base.tail_fraction, base.effective_rank_g, "--", color=".6")
    rank_ax.set_ylabel("effective rank", color=".5")
    panels = [
        (axes[0, 1], "gen_error_normalized", r"$E_{gen}/S$", "log"),
        (axes[0, 2], "r2_gen", r"$R^2_{gen}$", "linear"),
        (axes[1, 0], "acc_error_normalized", r"$E_{acc}/S$", "log"),
        (axes[1, 1], "r2_acc", r"$R^2_{acc}$", "symlog"),
        (axes[1, 2], "slope_acc", r"$\mathrm{slope}_{acc}$", "log"),
    ]
    for ratio, color in zip(ratios, colors):
        part = data[np.isclose(data.noise_signal_ratio, ratio)].sort_values("tail_fraction")
        label = rf"$\sigma={part.sigma.iloc[0]:.3g}$ ($\sigma^2/S={ratio:g}$)"
        for ax, metric, ylabel, scale in panels:
            ax.plot(part.tail_fraction, part[f"de_{metric}"], color=color, label=label)
            stat = r2_stat if metric == "r2_acc" else "mean"
            column = f"mc_{metric}_{stat}"
            if column in part:
                subset = part.iloc[::marker_every]
                ax.plot(subset.tail_fraction, subset[column], "o",
                        color=color, ms=4, mec=".2", mew=.3)
            ax.set_ylabel(ylabel)
            ax.set_title(ylabel)
            if scale == "symlog":
                ax.set_yscale(scale, linthresh=1)
            else:
                ax.set_yscale(scale)
    axes[1, 1].axhline(0, ls="--", color=".6", lw=.8)
    axes[0, 1].legend(fontsize=8, loc="best")
    for ax in axes.flat:
        ax.set_xscale("symlog", linthresh=1e-6, linscale=.6)
        ax.grid(alpha=.18)
    for ax in axes[1]:
        ax.set_xlabel(r"tail loading $t=\sin^2\theta$")
    fig.suptitle(f"Van Hateren disk teacher: p={p}; DE lines, MC circles")
    fig.tight_layout()
    return fig, axes


In [ ]:
fig, axes = plot_experiment(frame, p=P)
fig.savefig(run_dir / "metrics.png", dpi=180, bbox_inches="tight")
plt.show()


## 查看特征如何变化

显示实际 F 行在 population PC basis 中的能量，不是 pixel image。这里未保存完整 population eigenvectors，因此不能把 PC 坐标直接 reshape 为图像。F 的行是 pixel gradient 的 PC 表示；比较原始 norm 与归一化能量，避免归一化掩盖放大效应。


In [ ]:
FEATURE_ROWS = np.array([0, P//2, P-1])
FEATURE_T = np.array([0., 1e-4, 1e-2, 1.])
features = experiment.feature_rows_pc(family, s, FEATURE_ROWS, FEATURE_T)
display(pd.DataFrame(np.linalg.norm(features, axis=2),
                     index=FEATURE_ROWS+1, columns=FEATURE_T).rename_axis("feature row"))
fig, axes = plt.subplots(1, len(FEATURE_T), figsize=(15, 3.5), sharey=True)
for j, t in enumerate(FEATURE_T):
    for i, row in enumerate(FEATURE_ROWS):
        # Cumulative energy avoids log-binning artifacts at individual PCs.
        energy = features[i,j]**2
        axes[j].plot(np.arange(1, len(s)+1), np.cumsum(energy)/energy.sum(),
                     label=f"row {row+1}")
    axes[j].set(xscale="log", xlabel="PC rank", title=f"t={t:g}", ylim=(0,1.02))
    axes[j].grid(alpha=.2)
axes[0].set_ylabel("cumulative fraction of feature energy")
axes[0].legend()
fig.tight_layout()
fig.savefig(run_dir / "features.png", dpi=180)
plt.show()
np.savez_compressed(run_dir / "features.npz", rows=FEATURE_ROWS, t=FEATURE_T, features=features)
